# Qwen Q8 Chat on RunPod (chat-only)
Run from the repository root on the Pod (Jupyter terminal: `cd` to the unpacked bundle). Order: 1 → 5. No secrets are stored in this notebook; the Gradio login is typed at runtime (hidden input).
**Stopping the app here does not stop the Pod.** Stop the Pod with `scripts/pod.py stop <pod_id>` (REST v2) or the console.

In [ ]:
# Cell 1: pinned upstream source + hash-locked dependencies
import pathlib, subprocess, sys
REPO_DIR = pathlib.Path.cwd()
assert (REPO_DIR / "qmc_runpod").is_dir(), "run from the repository root"
sys.path.insert(0, str(REPO_DIR))
from qmc_runpod import layout, pins
P = layout.paths(); P["source"].mkdir(parents=True, exist_ok=True)
SRC = P["source"] / "qwen-multimodal-colab"
if not (SRC / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", pins.UPSTREAM_REPO, str(SRC)], check=True)
subprocess.run(["git", "-C", str(SRC), "fetch", "origin", pins.UPSTREAM_SHA], check=True)
subprocess.run(["git", "-C", str(SRC), "checkout", "-q", pins.UPSTREAM_SHA], check=True)
head = subprocess.run(["git", "-C", str(SRC), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
blob = subprocess.run(["git", "-C", str(SRC), "rev-parse", "HEAD:Qwen-Q8-Chat-Colab.ipynb"], capture_output=True, text=True, check=True).stdout.strip()
assert head == pins.UPSTREAM_SHA and blob == pins.UPSTREAM_NOTEBOOK_BLOB, "upstream pin mismatch"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--require-hashes", "-r", str(REPO_DIR / "requirements-runpod.lock.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
sys.path.insert(0, str(SRC / "src"))
print("upstream", head[:10], "python", sys.version.split()[0])

In [ ]:
# Cell 2: non-secret env, llama-server at the pinned commit, models at pinned revisions (SHA256-verified)
from qmc_runpod import layout, llama, models
ENV = layout.apply_env(ctx=8192, web_search="off")   # 8k smoke first; 32768 is a separate later run
SERVER = llama.install()
MODEL_PATHS, MODEL_RECORDS = models.fetch_pinned(layout.paths()["hf_cache"])
print(SERVER, [r["sha256_verified"] for r in MODEL_RECORDS])

In [ ]:
# Cell 3: start the chat-only UI. Needs BOTH login values (hidden input); refuses to start otherwise.
import getpass, os
os.environ["QMC_AUTH_USER"] = getpass.getpass("Gradio user: ")
os.environ["QMC_AUTH_PASSWORD"] = getpass.getpass("Gradio password: ")
from qmc_runpod import launch
APP = launch.launch(model_paths=MODEL_PATHS)
print(f"https://{os.environ.get('RUNPOD_POD_ID', '<pod-id>')}-{APP.cfg.server_port}.proxy.runpod.net/")

In [ ]:
# Cell 4: text-only measurements (numbers only; no prompt/answer text is recorded)
from qmc_runpod import smoke
PROMPTS = ["日本の四季を50字で説明して。", "1から10までの和は？", "Pythonでリストを逆順にする方法は？",
           "富士山の高さは？", "挨拶を一言。", "Gitのcommitとpushの違いは？",
           "味噌汁の基本の作り方を3行で。", "TCPとUDPの違いを一言で。", "今日の気分を一言で。", "素数とは？"]
turns = [smoke.run_turn(APP, p) for p in PROMPTS]
SUMMARY = smoke.summarize(turns); GPU = smoke.gpu_memory_mib()
print(SUMMARY, GPU)

In [ ]:
# Cell 5: allowlisted report + bundle. Download the .tar.gz from Jupyter; then stop the Pod from outside.
import datetime, platform
from qmc_runpod import report, pins, llama
rep = {
    "trial_id": "t" + datetime.datetime.now().strftime("%Y%m%d-%H%M"),
    "created_at_jst": datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).isoformat(timespec="seconds"),
    "upstream": {"sha": pins.UPSTREAM_SHA, "notebook_blob": pins.UPSTREAM_NOTEBOOK_BLOB},
    "llama_cpp": {"commit": pins.LLAMA_CPP_COMMIT, "version_line": llama.server_version(SERVER)},
    "models": [{k: r[k] for k in ("role", "file", "revision", "sha256_verified", "size_bytes")} for r in MODEL_RECORDS],
    "environment": {"gpu_name": GPU.get("gpu_name", "unknown"), "vram_mib": GPU.get("total_mib", 0),
                    "driver": GPU.get("driver", "unknown"), "python": platform.python_version()},
    "settings": {"ctx": int(ENV["QMC_CHAT_CTX"]), "thinking": False, "web_search": "off", "asr_device": "cpu",
                 "asr_model": "small", "tts": False, "share": False, "chat_only": True},
    "metrics": {k: v for k, v in {**SUMMARY, "vram_used_mib": GPU.get("used_mib")}.items() if v is not None},
}
path = report.write_report(rep, layout.paths()["runs"])
bundle = layout.paths()["runs"] / (path.stem + ".tar.gz")
report.make_bundle([path], bundle)
print(bundle)

In [ ]:
# Cell 6: stop the app and the model process. This does NOT stop or delete the Pod.
launch.stop_app(APP)